# Paper figures

Figures sized for the IEEE template, using the 7-day-block joint bootstrap intervals from notebook 09 (`joint_bootstrap_contrasts.csv`).
- Fig. 2: per-airport weather gain (`weather_ap_intervals_week`).
- Appendix: per-airport reference gain (`reference_ap_intervals_week`).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_DIR = PROJECT_ROOT / "outputs/final_evaluation"
FIGURE_DIR = RESULTS_DIR / "figures"
FAMILIES = {"logistic": "Logistic regression", "forest": "Random forest", "xgboost": "XGBoost"}

# Sized for the full text width of the IEEE two-column template (about 7.16 in), so fonts stay readable in print
plt.rcParams.update({"font.size": 8, "axes.titlesize": 9, "savefig.dpi": 300})

contrasts = pd.read_csv(RESULTS_DIR / "joint_bootstrap_contrasts.csv")
weekly = contrasts.loc[contrasts["scheme"].eq("week") & contrasts["group"].eq("weather")
                       & contrasts["airport"].ne("Equal-airport mean")].copy()
weekly["family"] = weekly["contrast"].str.split("|").str[1]
assert len(weekly) == 30 and weekly["n_boot"].eq(1000).all()
airports = sorted(weekly["airport"].unique())

fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.9), sharex=True, sharey=True, layout="constrained")
for ax, (family, label) in zip(axes, FAMILIES.items()):
    subset = weekly.loc[weekly["family"].eq(family)].set_index("airport").loc[airports]
    y = np.arange(len(airports))
    ax.hlines(y, subset["ci_lower"], subset["ci_upper"], color="#2878A5", linewidth=1.6)
    ax.scatter(subset["estimate"], y, color="#2878A5", s=14, zorder=3)
    ax.axvline(0, color="grey", linestyle="--", linewidth=0.8)
    ax.set_title(label)
    ax.set_yticks(y, airports)
    ax.grid(axis="x", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].invert_yaxis()
fig.supxlabel("AP difference: weather minus flight-only (unseen airports, 2025)", fontsize=8)
for extension in ["pdf", "png"]:
    fig.savefig(FIGURE_DIR / f"weather_ap_intervals_week.{extension}", bbox_inches="tight")
print("Saved weather_ap_intervals_week.pdf/.png")

# Appendix: per-airport reference gain (reference minus unseen), same 7-day-block intervals
reference = contrasts.loc[contrasts["scheme"].eq("week") & contrasts["group"].eq("reference")
                          & contrasts["airport"].ne("Equal-airport mean")].copy()
parts = reference["contrast"].str.split("|")
reference["family"], reference["variant"] = parts.str[1], parts.str[2]
assert len(reference) == 60
fig, axes = plt.subplots(2, 3, figsize=(7.16, 5.4), sharex=True, sharey=True, layout="constrained")
for row, variant in enumerate(["flight_only", "weather"]):
    for ax, (family, label) in zip(axes[row], FAMILIES.items()):
        subset = (reference.loc[reference["family"].eq(family) & reference["variant"].eq(variant)]
                  .set_index("airport").loc[airports])
        y = np.arange(len(airports))
        ax.hlines(y, subset["ci_lower"], subset["ci_upper"], color="#D97928", linewidth=1.6)
        ax.scatter(subset["estimate"], y, color="#D97928", s=14, zorder=3)
        ax.axvline(0, color="grey", linestyle="--", linewidth=0.8)
        ax.set_title(f"{label}, {variant.replace('_', '-')}")
        ax.set_yticks(y, airports)
        ax.grid(axis="x", alpha=0.2)
        ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].invert_yaxis()
fig.supxlabel("AP difference: reference minus unseen (2025)", fontsize=8)
for extension in ["pdf", "png"]:
    fig.savefig(FIGURE_DIR / f"reference_ap_intervals_week.{extension}", bbox_inches="tight")
print("Saved reference_ap_intervals_week.pdf/.png")
